# Silver Layer — Clean & Conform Construction Data
Dedupe, cast types. Keeps task label is_delayed; FE drops post-task leakage.

In [ ]:
from pyspark.sql.functions import (
    col, when, lit, to_date, row_number, current_timestamp
)
from pyspark.sql.window import Window

In [ ]:
# Clean subcontractors
df_s = spark.read.format('delta').table('bronze_subcontractors')
w = Window.partitionBy('subcontractor_id').orderBy(col('ingestion_timestamp').desc())
df_s = (
    df_s.withColumn('_rn', row_number().over(w)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('rating', col('rating').cast('double'))
    .withColumn('years_active', col('years_active').cast('int'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_s.write.mode('overwrite').format('delta').saveAsTable('silver_subcontractors')
print(f'silver_subcontractors: {spark.table("silver_subcontractors").count()} rows')

In [ ]:
# Clean projects
df_p = spark.read.format('delta').table('bronze_projects')
w2 = Window.partitionBy('project_id').orderBy(col('ingestion_timestamp').desc())
df_p = (
    df_p.withColumn('_rn', row_number().over(w2)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('budget', col('budget').cast('double'))
    .withColumn('planned_start_date', to_date('planned_start_date'))
    .withColumn('planned_end_date', to_date('planned_end_date'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_p.write.mode('overwrite').format('delta').saveAsTable('silver_projects')
print(f'silver_projects: {spark.table("silver_projects").count()} rows')

In [ ]:
# Clean tasks (keep label is_delayed)
df_t = spark.read.format('delta').table('bronze_tasks')
w3 = Window.partitionBy('task_id').orderBy(col('ingestion_timestamp').desc())
df_t = (
    df_t.withColumn('_rn', row_number().over(w3)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('planned_start_date', to_date('planned_start_date'))
    .withColumn('planned_end_date', to_date('planned_end_date'))
    .withColumn('planned_duration_days', col('planned_duration_days').cast('int'))
    .withColumn('schedule_variance_days', col('schedule_variance_days').cast('int'))
    .withColumn('is_delayed', col('is_delayed').cast('int'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_t.write.mode('overwrite').format('delta').saveAsTable('silver_tasks')
print(f'silver_tasks: {spark.table("silver_tasks").count()} rows')

In [ ]:
# Clean cost ledger
df_c = spark.read.format('delta').table('bronze_cost_ledger')
df_c = (
    df_c
    .withColumn('entry_date', to_date('entry_date'))
    .withColumn('planned_cost', col('planned_cost').cast('double'))
    .withColumn('actual_cost', col('actual_cost').cast('double'))
    .withColumn('cost_variance_pct', col('cost_variance_pct').cast('double'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_c.write.mode('overwrite').format('delta').saveAsTable('silver_cost_ledger')
print(f'silver_cost_ledger: {spark.table("silver_cost_ledger").count()} rows')